# 001 · Coleta — Observatório da Arborização Urbana de SP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/micadw/arborizacao-sp/blob/main/notebooks/001_coleta.ipynb)

> Projeto pessoal com dados abertos; não representa posição oficial da PMSP.

Este notebook baixa as fontes do catálogo (`docs/01_catalogo_fontes.md`) para o **Drive** e registra cada download (data, URL, hash). Rode com *Ambiente de execução → Reiniciar e executar tudo*.

## 0. Ambiente: Drive, repositório e pacotes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
# Ajuste para o caminho do projeto no seu Drive (Nível 4/5 das Diretrizes)
DATA = Path('/content/drive/MyDrive/arborizacao-sp/data')
RAW = DATA / 'raw'; RAW.mkdir(parents=True, exist_ok=True)
REGISTRO = DATA / 'registro_downloads.csv'

In [ ]:
# Repositório privado: o Colab pede autorização do GitHub na primeira vez
!git clone -q https://github.com/micadw/arborizacao-sp.git /content/arborizacao-sp || (cd /content/arborizacao-sp && git pull -q)
import sys; sys.path.append('/content/arborizacao-sp')
!pip -q install geopandas pyogrio
from src import coleta

## 1. Portal de dados abertos (CKAN)
Procura datasets relacionados e lista os recursos do SP156.

In [ ]:
for termo in ['arvore', 'arborizacao', 'poda', 'sp156', 'cobertura vegetal']:
    print(f'\n== {termo} ==')
    display(coleta.ckan_buscar(termo).head(15))

In [ ]:
rec156 = coleta.ckan_recursos('dados-do-sp156')
rec156

In [ ]:
# Baixa os arquivos do SP156 de 2015 em diante (ajuste o filtro após olhar a tabela acima)
sel = rec156[rec156['formato'].str.upper().isin(['CSV', 'ZIP'])]
for _, r in sel.iterrows():
    destino = RAW / 'sp156' / Path(r['url']).name
    if not destino.exists():
        coleta.baixar(r['url'], destino)
        coleta.registrar_download(REGISTRO, 'catalogo#3 SP156', r['url'], destino)
print('ok')

## 2. GeoSampa (WFS)
Primeiro confirme a URL do serviço e os nomes das camadas — o catálogo marca isso como *a confirmar*.

In [ ]:
camadas = coleta.wfs_camadas()
[c for c in camadas if any(k in c.lower() for k in ['arvore', 'vegeta', 'distrito', 'subpref', 'quadra'])]

In [ ]:
# Substitua pelos nomes confirmados acima
CAMADAS = {
    'arvores': 'geoportal:arvore',
    'distritos': 'geoportal:distrito_municipal',
    'subprefeituras': 'geoportal:subprefeitura',
}
gdfs = {}
for nome, camada in CAMADAS.items():
    try:
        gdfs[nome] = coleta.wfs_camada(camada)
        destino = RAW / 'geosampa' / f'{nome}.gpkg'
        destino.parent.mkdir(parents=True, exist_ok=True)
        gdfs[nome].to_file(destino, driver='GPKG')
        coleta.registrar_download(REGISTRO, f'GeoSampa {camada}', coleta.GEOSAMPA_WFS, destino)
        print(nome, len(gdfs[nome]))
    except Exception as e:
        print('FALHOU', nome, camada, e)

## 3. Orçamento (API-SOF)
Gere o token gratuito na Vitrine de APIs da Prefeitura e salve em *Secrets* do Colab como `SOF_TOKEN`.

In [ ]:
from google.colab import userdata
token = userdata.get('SOF_TOKEN')
desp = coleta.sof_despesas(2026, token)
# Filtrar ações de plantio, manejo e arborização (ajustar após inspecionar)
mask = desp.astype(str).apply(lambda c: c.str.contains('ARBORI|PLANTIO|MANEJO|ÁRVORE|ARVORE', case=False)).any(axis=1)
desp_arb = desp[mask]
desp_arb.to_csv(RAW / 'sof_arborizacao_2026.csv', index=False)
desp_arb.head()

## 4. Primeira agregação: árvores viárias por distrito

In [ ]:
if {'arvores', 'distritos'} <= gdfs.keys():
    chave = [c for c in gdfs['distritos'].columns if 'nome' in c.lower() or 'nm_' in c.lower()][0]
    arv_dist = coleta.contar_por_territorio(gdfs['arvores'], gdfs['distritos'], chave)
    display(arv_dist.sort_values('n', ascending=False).head(10))
    out = gdfs['distritos'].merge(arv_dist, on=chave, how='left').to_crs('EPSG:4326')
    out.to_file('/content/arborizacao-sp/site_data/distrito.geojson', driver='GeoJSON')

## 5. Salvar no GitHub
*Arquivo → Salvar uma cópia no GitHub* com mensagem `feat(coleta): primeira execução`. Os dados brutos ficam só no Drive; apenas `site_data/` (leve) pode ir para o repositório.